## Module requires

In [1]:
pip install cdsapi xarray netCDF4

Note: you may need to restart the kernel to use updated packages.


## Documentation

In [ ]:
"""
era5_land.py
============
WAC PhD Pipeline — ERA5-Land Monthly Download Script
Candidate: TOURE Seydou | 2026

Downloads ERA5-Land monthly-averaged NetCDF files for the West African Craton
(WAC) bounding box, covering all 21 gold mine site locations.

Variables downloaded
--------------------
  • 2m_temperature      (t2m)  [K]  → converted to °C at extraction stage
  • skin_temperature    (skt)  [K]  → converted to °C at extraction stage
  • total_precipitation (tp)   [m]  → converted to mm at extraction stage

Resolution  : 0.1° (~9 km)   ← ERA5-Land native resolution
Product     : reanalysis-era5-land-monthly-means
Period      : 1981–2025  (one NetCDF file per year, all 12 months)
Output dir  : ./outputs/netcdf/era5land/

WAC Bounding Box
----------------
  North: 16°N  |  West: 12°W  |  South: 3°N  |  East: 3°E

CDS API Credentials (new endpoint — cdsapi >= 0.6.0)
-----------------------------------------------------
  url : https://cds.climate.copernicus.eu/api
  key : c2629c3f-a951-48f6-b356-5c59f3c1f0c5

  Credentials are passed directly to cdsapi.Client() — no .cdsapirc needed.

IMPORTANT — Terms of Use
-------------------------
Before running, you must accept the ERA5-Land licence ONCE on the CDS portal:
  https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-monthly-means
  → Click "Terms of use" at the bottom → Accept

Usage
-----
  # Download full period (1981–2025)
  python era5_land.py

  # Download a specific range
  python era5_land.py --year-start 1995 --year-end 2025

  # Re-download files even if they exist
  python era5_land.py --overwrite

  # Verify already-downloaded files
  python era5_land.py --verify

Reference
---------
Muñoz-Sabater, J. et al. (2021). ERA5-Land: A state-of-the-art global
reanalysis dataset for land applications. Earth Syst. Sci. Data, 13,
4349–4383. https://doi.org/10.5194/essd-13-4349-2021
"""

# Package  and module require

In [ ]:
import argparse
import logging
import sys
import time
from pathlib import Path
from datetime import datetime

## Logging

In [ ]:
# Path.cwd() works both in Jupyter notebooks and in .py scripts
try:
    BASE_DIR = Path(__file__).parent.resolve()
except NameError:
    BASE_DIR = Path.cwd()
LOG_DIR  = BASE_DIR / "outputs" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_file = LOG_DIR / f"era5land_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(log_file, encoding="utf-8"),
    ],
)
logger = logging.getLogger("ERA5_LAND")

## CONFIGURATION 

In [ ]:
# CDS API credentials (new endpoint — do NOT add /v2)
CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = "c2629c3f-a951-48f6-b356-5c59f3c1f0c5"

# CDS dataset identifier
DATASET = "reanalysis-era5-land-monthly-means"

# Variables to download
VARIABLES = [
    "2m_temperature",       # T2m — primary temperature predictor
    "skin_temperature",     # Ts  — surface radiative/skin temperature
    "total_precipitation",  # TP  — backup precipitation (cross-check vs CHIRPS)
]

# WAC bounding box [North, West, South, East] — covers all 21 mine sites
# Burkina Faso / Côte d'Ivoire / Ghana / Mali
BBOX = [16.0, -12.0, 3.0, 3.0]   # [N, W, S, E]

# Study period
YEAR_START = 1981   # ERA5-Land starts 1981
YEAR_END   = 2025   # Last complete calendar year

# Output directory
OUT_DIR = BASE_DIR / "outputs" / "netcdf" / "era5land"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## DOWNLOAD FUNCTIONS

In [ ]:

def _make_client():
    """
    Initialise a cdsapi.Client with explicit credentials.
    Passing url and key directly avoids any ~/.cdsapirc conflict.
    """
    try:
        import cdsapi
    except ImportError:
        raise ImportError(
            "cdsapi not found. Install it with:\n"
            "  pip install cdsapi"
        )
    return cdsapi.Client(url=CDS_URL, key=CDS_KEY)


def download_year(client, year: int, out_file: Path) -> bool:
    """
    Submit one CDS request for a single year (all 12 months, 3 variables).

    Parameters
    ----------
    client   : cdsapi.Client  Authenticated CDS client
    year     : int            Calendar year to download
    out_file : Path           Destination NetCDF file path

    Returns
    -------
    bool  True if download succeeded, False otherwise
    """
    request = {
        # Product type — monthly averaged reanalysis (not hourly)
        "product_type": ["monthly_averaged_reanalysis"],

        # Variables — all three in one request (CDS merges them)
        "variable": VARIABLES,

        # Year and all 12 months — CDS returns one file with 12 time steps
        "year":  [str(year)],
        "month": [f"{m:02d}" for m in range(1, 13)],

        # Required for monthly means product
        "time": ["00:00"],

        # Spatial subset — WAC domain [N, W, S, E]
        "area": BBOX,

        # Output format — netcdf (NetCDF-4, CF-compliant)
        # Use "netcdf_legacy" only if you need NetCDF-3 compatibility
        "data_format": "netcdf",

        # Return as a single unarchived file (not a .tar.gz)
        "download_format": "unarchived",
    }

    logger.info(f"  Submitting CDS request for {year}…")
    logger.debug(f"  Request payload: {request}")

    try:
        client.retrieve(DATASET, request, str(out_file))
        size_mb = out_file.stat().st_size / 1e6
        logger.info(f"  ✓ {year} → {out_file.name}  ({size_mb:.1f} MB)")
        return True

    except Exception as exc:
        logger.error(f"  ✗ {year} — request failed: {exc}")
        # Remove partial/empty file if it was created
        if out_file.exists() and out_file.stat().st_size == 0:
            out_file.unlink()
        return False


def download_era5_land(year_start: int = YEAR_START,
                       year_end:   int = YEAR_END,
                       overwrite:  bool = False) -> list:
    """
    Main download loop — one NetCDF file per year.

    Each file contains:
      • 12 monthly time steps
      • 3 variables: t2m, skt, tp
      • WAC spatial extent at 0.1° resolution
      • Approximate file size: 5–15 MB per year

    Parameters
    ----------
    year_start : int   First year (min 1981)
    year_end   : int   Last year
    overwrite  : bool  Re-download even if the file already exists

    Returns
    -------
    list[Path]  All successfully downloaded (or already-existing) files
    """
    logger.info("=" * 65)
    logger.info("ERA5-Land Download — WAC PhD Pipeline")
    logger.info("=" * 65)
    logger.info(f"Dataset  : {DATASET}")
    logger.info(f"Variables: {', '.join(VARIABLES)}")
    logger.info(f"Period   : {year_start}–{year_end}  ({year_end - year_start + 1} years)")
    logger.info(f"Bbox     : N={BBOX[0]} W={BBOX[1]} S={BBOX[2]} E={BBOX[3]}")
    logger.info(f"Output   : {OUT_DIR}")
    logger.info("=" * 65)

    client    = _make_client()
    success   = []
    skipped   = []
    failed    = []
    t_start   = time.time()

    for year in range(year_start, year_end + 1):
        out_file = OUT_DIR / f"era5land_{year}.nc"

        # ── Skip if already downloaded ─────────────────────────────────────
        if out_file.exists() and not overwrite:
            size_mb = out_file.stat().st_size / 1e6
            logger.info(f"[{year}] Already exists ({size_mb:.1f} MB) — skipping. "
                        f"Use --overwrite to re-download.")
            skipped.append(out_file)
            success.append(out_file)
            continue

        logger.info(f"[{year}] Downloading…")
        ok = download_year(client, year, out_file)

        if ok:
            success.append(out_file)
        else:
            failed.append(year)
            # Brief pause before next request to avoid hammering the API
            time.sleep(5)

    # ── Summary ───────────────────────────────────────────────────────────────
    elapsed = time.time() - t_start
    total   = year_end - year_start + 1

    logger.info("")
    logger.info("=" * 65)
    logger.info("DOWNLOAD SUMMARY")
    logger.info("=" * 65)
    logger.info(f"  Total years requested : {total}")
    logger.info(f"  Successfully available: {len(success)}  "
                f"({len(skipped)} already existed, "
                f"{len(success) - len(skipped)} newly downloaded)")
    logger.info(f"  Failed                : {len(failed)}")
    if failed:
        logger.warning(f"  Failed years: {failed}")
    logger.info(f"  Elapsed time          : {elapsed / 60:.1f} min")
    logger.info(f"  Output directory      : {OUT_DIR}")
    logger.info(f"  Log file              : {log_file}")
    logger.info("=" * 65)

    return success

## VERIFICATION

In [ ]:
def verify_downloads(year_start: int = YEAR_START,
                     year_end:   int = YEAR_END) -> None:
    """
    Scan the output directory and validate each NetCDF file with xarray.
    Prints a summary table of coverage, variables found, and file sizes.
    """
    try:
        import xarray as xr
    except ImportError:
        logger.error("xarray not installed. Run: pip install xarray netCDF4")
        return

    logger.info("")
    logger.info("=" * 65)
    logger.info("ERA5-Land File Verification")
    logger.info("=" * 65)

    total      = year_end - year_start + 1
    found      = 0
    valid      = 0
    total_mb   = 0.0

    print(f"\n{'Year':<6} {'File':<28} {'Size (MB)':<12} {'Time steps':<12} {'Variables'}")
    print("─" * 85)

    for year in range(year_start, year_end + 1):
        f = OUT_DIR / f"era5land_{year}.nc"

        if not f.exists():
            print(f"{year:<6} {'— NOT FOUND —':<28}")
            continue

        found    += 1
        size_mb   = f.stat().st_size / 1e6
        total_mb += size_mb

        try:
            ds        = xr.open_dataset(f, engine="netcdf4")
            n_times   = len(ds.time) if "time" in ds.dims else "?"
            var_names = list(ds.data_vars)
            ds.close()
            status = "✓"
            valid += 1
            print(f"{year:<6} {f.name:<28} {size_mb:<12.1f} {str(n_times):<12} "
                  f"{', '.join(var_names)}")
        except Exception as exc:
            print(f"{year:<6} {f.name:<28} {size_mb:<12.1f} {'ERROR':<12} {exc}")

    print("─" * 85)
    print(f"\nSummary: {valid}/{total} years valid | "
          f"{found - valid} corrupted | "
          f"{total - found} missing | "
          f"Total size: {total_mb:.0f} MB ({total_mb/1024:.2f} GB)")
    print(f"Output: {OUT_DIR}\n")

In [1]:
#CLIPPING
# ══════════════════════════════════════════════════════════════════════════════

def main():
    parser = argparse.ArgumentParser(
        description="ERA5-Land monthly downloader for the WAC PhD pipeline",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument(
        "--year-start", type=int, default=YEAR_START,
        help=f"First year to download (default: {YEAR_START})",
    )
    parser.add_argument(
        "--year-end", type=int, default=YEAR_END,
        help=f"Last year to download (default: {YEAR_END})",
    )
    parser.add_argument(
        "--overwrite", action="store_true",
        help="Re-download files even if they already exist on disk",
    )
    parser.add_argument(
        "--verify", action="store_true",
        help="Verify already-downloaded files (no download)",
    )
    args = parser.parse_args()

    if args.verify:
        verify_downloads(args.year_start, args.year_end)
    else:
        files = download_era5_land(
            year_start=args.year_start,
            year_end=args.year_end,
            overwrite=args.overwrite,
        )
        print(f"\n✓ {len(files)} ERA5-Land files available in:\n  {OUT_DIR}")
        print(f"  Log → {log_file}")
        print("\nNext step:")
        print("  python era5_land.py --verify     ← inspect downloaded files")
        print("  Then proceed to CHIRPS download.")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--year-start YEAR_START] [--year-end YEAR_END] [--overwrite] [--verify]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\LENOVO\AppData\Roaming\jupyter\runtime\kernel-6cf13e87-0ff3-4481-bcd0-21e003e6d29c.json


SystemExit: 2

C:\Users\LENOVO\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
# Cellule 1 — Importer les fonctions
exec(open("era5_land.py").read())

FileNotFoundError: [Errno 2] No such file or directory: 'era5_land.py'